In [ ]:
%pip install torch


In [ ]:

import os, glob, pickle, random, time, math, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from scipy.stats import chi2

# ---------------- CONFIG (fix these BEFORE the final comparison) ----------------
SEEDS = [42]          # model-init seeds. Use e.g. [0, 1, 2] for mean +/- std (3x training time)
SPLIT_SEED = 2024     # fixed seed for the train/val/test split (same for every model and seed)
TRAIN_FRAC, VAL_FRAC = 0.6, 0.2          # test = remaining 0.2
EPOCHS = 40
BATCH = 256
LR = 1e-3
PATIENCE = 8          # early stopping on validation loss
DROPOUT = 0.5
PAIRS = [("QAM16", "QAM64"), ("QPSK", "8PSK")]   # RadioML 2016.10a uses the names QAM16 / QAM64
LOW_SNR_DEMO, HIGH_SNR_DEMO = -4, 10             # SNRs used for Grad-CAM comparisons
# -------------------------------------------------------------------------------

BASE = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUT = BASE / "amc_outputs"
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "| outputs ->", OUT.resolve())


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def savefig(name):
    plt.tight_layout()
    plt.savefig(OUT / name, dpi=150)
    plt.show()

In [ ]:
DATA_PATH = os.environ.get("RADIOML_DATA_PATH", str(Path("data") / "RML2016.10a_dict.pkl"))

if not os.path.isfile(DATA_PATH):
    raise FileNotFoundError(f"File not found: {DATA_PATH}")
print("Loading:", DATA_PATH)
with open(DATA_PATH, "rb") as f:
    d = pickle.load(f, encoding="latin1")   # only load pickles from sources you trust

MODS = sorted({k[0] for k in d})
SNRS = sorted({k[1] for k in d})
print("Modulations:", MODS)
print("SNRs:", SNRS)
assert len(MODS) == 11 and len(SNRS) == 20, "Unexpected dataset: expected 11 classes and 20 SNR levels"
assert SNRS[0] == -20 and SNRS[-1] == 18
for p in PAIRS:
    assert all(m in MODS for m in p), f"Pair {p} not in {MODS}"

X_list, y_list, s_list = [], [], []
for m in MODS:
    for s in SNRS:
        a = d[(m, s)]
        assert a.shape[1:] == (2, 128), a.shape
        X_list.append(a)
        y_list += [MODS.index(m)] * len(a)
        s_list += [s] * len(a)
X = np.vstack(X_list).astype(np.float32)      # (N, 2, 128): channel 0 = I, channel 1 = Q
y = np.array(y_list, dtype=np.int64)
snr = np.array(s_list, dtype=np.int64)
del d, X_list
print("X:", X.shape, "| samples per (class, SNR):", len(X) // (len(MODS) * len(SNRS)))

In [ ]:
rms = np.sqrt((X ** 2).sum(axis=1).mean(axis=1))          # (N,)
X /= (rms[:, None, None] + 1e-8)

rng = np.random.RandomState(SPLIT_SEED)
tr, va, te = [], [], []
for m in range(len(MODS)):
    for s in SNRS:
        idx = np.where((y == m) & (snr == s))[0]
        rng.shuffle(idx)
        n_tr, n_va = int(len(idx) * TRAIN_FRAC), int(len(idx) * VAL_FRAC)
        tr += list(idx[:n_tr]); va += list(idx[n_tr:n_tr + n_va]); te += list(idx[n_tr + n_va:])
idx_tr, idx_va, idx_te = map(np.array, (tr, va, te))
assert not (set(idx_tr) & set(idx_va)) and not (set(idx_tr) & set(idx_te)) and not (set(idx_va) & set(idx_te))
np.savez(OUT / "split_indices.npz", train=idx_tr, val=idx_va, test=idx_te)
print(f"train {len(idx_tr)} | val {len(idx_va)} | test {len(idx_te)}")

X_te, y_te, snr_te = X[idx_te], y[idx_te], snr[idx_te]


def make_loader(idx, shuffle, seed=0):
    ds = TensorDataset(torch.from_numpy(X[idx]), torch.from_numpy(y[idx]))
    g = torch.Generator().manual_seed(seed)
    return DataLoader(ds, batch_size=BATCH, shuffle=shuffle, generator=g if shuffle else None,
                      pin_memory=(DEVICE.type == "cuda"))


val_loader = make_loader(idx_va, shuffle=False)

In [ ]:
class ConvFeatures(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(2, 64, 7, padding=3), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),     # 128 -> 64
            nn.Conv1d(64, 128, 5, padding=2), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),  # 64 -> 32
            nn.Conv1d(128, 128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),                  # Grad-CAM target
        )

    def forward(self, x):
        return self.net(x)          # (B, 128, 32)


class LightSelfAttention(nn.Module):
    def __init__(self, dim=128, dk=32):
        super().__init__()
        self.q, self.k, self.v = nn.Linear(dim, dk), nn.Linear(dim, dk), nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)
        self.dk = dk
        self.last_attn = None

    def forward(self, x):           # x: (B, T, dim)
        q, k, v = self.q(x), self.k(x), self.v(x)
        w = torch.softmax(q @ k.transpose(1, 2) / math.sqrt(self.dk), dim=-1)   # (B, T, T)
        self.last_attn = w.detach()
        return self.norm(x + w @ v)


class AMCNet(nn.Module):
    def __init__(self, use_attention, n_classes=11, T=32, C=128, dropout=DROPOUT):
        super().__init__()
        self.features = ConvFeatures()
        self.attn = LightSelfAttention(C) if use_attention else None
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(C * T, 128), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(128, n_classes))

    def classify(self, f):          # f: (B, C, T) conv feature maps
        if self.attn is not None:
            f = self.attn(f.transpose(1, 2)).transpose(1, 2)
        return self.head(f)

    def forward(self, x):
        return self.classify(self.features(x))


for nm, ua in [("CNN", False), ("CNN+Attn", True)]:
    print(nm, "parameters:", sum(p.numel() for p in AMCNet(ua).parameters()))

In [ ]:
def run_epoch(model, loader, opt=None):
    train = opt is not None
    model.train(train)
    tot_loss = tot_corr = n = 0
    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            out = model(xb)
            loss = F.cross_entropy(out, yb)
            if train:
                opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            tot_loss += loss.item() * len(yb)
            tot_corr += (out.argmax(1) == yb).sum().item()
            n += len(yb)
    return tot_loss / n, tot_corr / n


def train_model(name, use_attention, seed):
    set_seed(seed)
    model = AMCNet(use_attention).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=3)
    train_loader = make_loader(idx_tr, shuffle=True, seed=seed)
    ckpt = OUT / f"{name.replace('+', '_')}_seed{seed}.pt"
    best, wait, hist = float("inf"), 0, []
    t0 = time.time()
    for ep in range(1, EPOCHS + 1):
        trl, tra = run_epoch(model, train_loader, opt)
        vl, vacc = run_epoch(model, val_loader)
        sched.step(vl)
        hist.append(dict(epoch=ep, train_loss=trl, train_acc=tra, val_loss=vl, val_acc=vacc))
        print(f"[{name} s{seed}] ep {ep:02d} | train {trl:.3f}/{tra:.3f} | val {vl:.3f}/{vacc:.3f} | {time.time()-t0:.0f}s")
        if vl < best - 1e-4:
            best, wait = vl, 0
            torch.save(model.state_dict(), ckpt)
        else:
            wait += 1
            if wait >= PATIENCE:
                print("  early stopping"); break
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    return model, pd.DataFrame(hist)


@torch.no_grad()
def predict_probs(model, X_np):
    model.eval()
    out = []
    for i in range(0, len(X_np), 1024):
        out.append(model(torch.from_numpy(X_np[i:i + 1024]).to(DEVICE)).softmax(1).cpu())
    return torch.cat(out).numpy()


def snr_metrics(pred):
    corr = pred == y_te
    return dict(per_snr=np.array([corr[snr_te == s].mean() for s in SNRS]),
                aggregate=corr.mean(), low=corr[snr_te < 0].mean(), high=corr[snr_te >= 0].mean())


RUNS = {}
for name, use_attn in [("CNN", False), ("CNN+Attn", True)]:
    RUNS[name] = []
    for seed in SEEDS:
        model, hist = train_model(name, use_attn, seed)
        pred = predict_probs(model, X_te).argmax(1)
        RUNS[name].append(dict(model=model, hist=hist, pred=pred, metrics=snr_metrics(pred)))
        print(f"==> {name} seed {seed}: aggregate {RUNS[name][-1]['metrics']['aggregate']*100:.2f}%")

In [ ]:
rows = []
for name, runs in RUNS.items():
    def ms(key):
        v = np.array([r["metrics"][key] * 100 for r in runs])
        return f"{v.mean():.2f} +/- {v.std():.2f}" if len(v) > 1 else f"{v.mean():.2f}"
    rows.append(dict(Model=name, Params=sum(p.numel() for p in runs[0]["model"].parameters()),
                     **{"Aggregate (%)": ms("aggregate"), "Low SNR <0 dB (%)": ms("low"), "High SNR >=0 dB (%)": ms("high")}))
summary = pd.DataFrame(rows)
summary.to_csv(OUT / "summary_metrics.csv", index=False)
display(summary) if "display" in globals() else print(summary)

per_snr = pd.DataFrame({name: np.mean([r["metrics"]["per_snr"] for r in runs], axis=0) * 100
                        for name, runs in RUNS.items()}, index=SNRS)
per_snr.index.name = "SNR_dB"
per_snr.to_csv(OUT / "per_snr_accuracy.csv")
print(per_snr.round(2).to_string())

plt.figure(figsize=(8, 5))
for name, runs in RUNS.items():
    arr = np.array([r["metrics"]["per_snr"] * 100 for r in runs])
    plt.plot(SNRS, arr.mean(0), marker="o", label=name)
    if len(arr) > 1:
        plt.fill_between(SNRS, arr.mean(0) - arr.std(0), arr.mean(0) + arr.std(0), alpha=0.2)
plt.axvline(0, ls="--", c="gray"); plt.xlabel("SNR (dB)"); plt.ylabel("Accuracy (%)")
plt.title("Accuracy vs SNR (test set)"); plt.grid(alpha=0.3); plt.legend()
savefig("accuracy_vs_snr.png")

# Paired significance test (McNemar) on the first seed, same test samples
p_cnn, p_att = RUNS["CNN"][0]["pred"], RUNS["CNN+Attn"][0]["pred"]
for label, mask in [("all SNR", np.ones_like(y_te, bool)), ("low SNR (<0 dB)", snr_te < 0), ("high SNR (>=0 dB)", snr_te >= 0)]:
    c1, c2 = p_cnn[mask] == y_te[mask], p_att[mask] == y_te[mask]
    b, c = int((c1 & ~c2).sum()), int((~c1 & c2).sum())
    pval = chi2.sf((abs(b - c) - 1) ** 2 / (b + c), 1) if b + c > 0 else 1.0
    print(f"McNemar {label:18s}: CNN-only-correct={b}, Attn-only-correct={c}, p={pval:.4g}")

In [ ]:
def cm_for(pred, mask):
    return confusion_matrix(y_te[mask], pred[mask], labels=list(range(len(MODS))))


fig, axes = plt.subplots(2, 2, figsize=(14, 12))
for r, (name, runs) in enumerate(RUNS.items()):
    for c, (lab, mask) in enumerate([("SNR < 0 dB", snr_te < 0), ("SNR >= 0 dB", snr_te >= 0)]):
        cm = cm_for(runs[0]["pred"], mask)
        cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
        ax = axes[r, c]
        ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(len(MODS))); ax.set_xticklabels(MODS, rotation=60, ha="right", fontsize=8)
        ax.set_yticks(range(len(MODS))); ax.set_yticklabels(MODS, fontsize=8)
        ax.set_title(f"{name} | {lab}"); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        for i in range(len(MODS)):
            for j in range(len(MODS)):
                if cmn[i, j] >= 0.05:
                    ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=6,
                            color="white" if cmn[i, j] > 0.5 else "black")
savefig("confusion_matrices.png")

print("Top confusion pairs (symmetric error counts, first seed):")
for name, runs in RUNS.items():
    for lab, mask in [("low SNR", snr_te < 0), ("high SNR", snr_te >= 0)]:
        cm = cm_for(runs[0]["pred"], mask)
        pairs = sorted(((cm[i, j] + cm[j, i], MODS[i], MODS[j])
                        for i in range(len(MODS)) for j in range(i + 1, len(MODS))), reverse=True)[:5]
        print(f"  {name:9s} {lab:8s}:", ", ".join(f"{a}<->{b} ({n})" for n, a, b in pairs))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, runs in RUNS.items():
    h = runs[0]["hist"]
    axes[0].plot(h.epoch, h.val_loss, label=f"{name} val"); axes[0].plot(h.epoch, h.train_loss, ls="--", label=f"{name} train")
    axes[1].plot(h.epoch, h.val_acc, label=f"{name} val"); axes[1].plot(h.epoch, h.train_acc, ls="--", label=f"{name} train")
axes[0].set_title("Loss"); axes[1].set_title("Accuracy")
for a in axes: a.set_xlabel("Epoch"); a.grid(alpha=0.3); a.legend()
savefig("training_curves.png")

In [ ]:
def grad_cam(model, x, target=None):
    """x: (B,2,128) tensor on DEVICE. Returns (cam (B,128) in [0,1], predicted class (B,), softmax probs (B,11))."""
    model.eval()
    with torch.enable_grad():
        feats = model.features(x)
        feats.retain_grad()
        logits = model.classify(feats)
        pred = logits.argmax(1)
        tgt = pred if target is None else target
        score = logits.gather(1, tgt[:, None]).sum()
        model.zero_grad(set_to_none=True)
        score.backward()
        w = feats.grad.mean(dim=2, keepdim=True)                 # (B, C, 1)
        cam = F.relu((w * feats).sum(1, keepdim=True))           # (B, 1, T)
        cam = F.interpolate(cam, size=x.shape[-1], mode="linear", align_corners=False)[:, 0]
        cam = cam / (cam.amax(1, keepdim=True) + 1e-8)
    return cam.detach().cpu().numpy(), pred.detach(), logits.softmax(1).detach()


def pair_groups(pred, pair, snr_val):
    A, B = [MODS.index(p) for p in pair]
    pool = np.where((snr_te == snr_val) & np.isin(y_te, [A, B]))[0]
    right = pool[pred[pool] == y_te[pool]]
    wrong = pool[(pred[pool] != y_te[pool]) & np.isin(pred[pool], [A, B])]   # confused with the other class of the pair
    return right, wrong


def plot_examples(model, mname, pred, pair, snr_val, n=3):
    right, wrong = pair_groups(pred, pair, snr_val)
    rs = np.random.RandomState(0)
    fig, axes = plt.subplots(2, n, figsize=(5 * n, 6))
    for r, (grp, lab) in enumerate([(right, "correct"), (wrong, "wrong (pair confusion)")]):
        sel = rs.choice(grp, min(n, len(grp)), replace=False) if len(grp) else []
        if len(sel):
            cam, _, _ = grad_cam(model, torch.from_numpy(X_te[sel]).to(DEVICE))
        for c in range(n):
            ax = axes[r, c]
            if c >= len(sel):
                ax.axis("off"); continue
            i = sel[c]
            ax.plot(X_te[i, 0], lw=0.8, label="I"); ax.plot(X_te[i, 1], lw=0.8, label="Q")
            ax2 = ax.twinx()
            ax2.fill_between(range(X_te.shape[-1]), cam[c], alpha=0.3, color="tab:red")
            ax2.set_ylim(0, 1.05); ax2.set_ylabel("Grad-CAM", fontsize=8)
            ax.set_title(f"{lab}\ntrue={MODS[y_te[i]]}  pred={MODS[pred[i]]}", fontsize=9)
            if r == 0 and c == 0: ax.legend(fontsize=7, loc="upper right")
    fig.suptitle(f"{mname} | {pair[0]} vs {pair[1]} | SNR = {snr_val} dB", y=1.01)
    savefig(f"gradcam_examples_{mname.replace('+','_')}_{pair[0]}_{pair[1]}_snr{snr_val}.png")


def mean_cam(model, idx):
    if len(idx) == 0:
        return None
    cams = []
    for i in range(0, len(idx), 512):
        c, _, _ = grad_cam(model, torch.from_numpy(X_te[idx[i:i + 512]]).to(DEVICE))
        cams.append(c)
    return np.concatenate(cams).mean(0)


# (a) individual examples: correct vs wrong, high vs low SNR, both models
for name, runs in RUNS.items():
    for pair in PAIRS:
        for sv in (HIGH_SNR_DEMO, LOW_SNR_DEMO):
            plot_examples(runs[0]["model"], name, runs[0]["pred"], pair, sv)

# (b) average saliency profile per condition
for pair in PAIRS:
    fig, axes = plt.subplots(1, len(RUNS), figsize=(6 * len(RUNS), 4), sharey=True)
    for ax, (name, runs) in zip(np.atleast_1d(axes), RUNS.items()):
        for sv, sl in [(HIGH_SNR_DEMO, "high"), (LOW_SNR_DEMO, "low")]:
            right, wrong = pair_groups(runs[0]["pred"], pair, sv)
            for grp, gl, ls in [(right, "correct", "-"), (wrong, "wrong", "--")]:
                mc = mean_cam(runs[0]["model"], grp)
                if mc is not None:
                    ax.plot(mc, ls=ls, label=f"{sl} SNR ({sv} dB), {gl} (n={len(grp)})")
        ax.set_title(f"{name}: mean Grad-CAM, {pair[0]} vs {pair[1]}"); ax.set_xlabel("Time sample"); ax.grid(alpha=0.3)
        ax.legend(fontsize=7)
    savefig(f"gradcam_mean_{pair[0]}_{pair[1]}.png")

In [ ]:
def deletion_test(model, snr_val, frac=0.25, seed=0):
    idx = np.where(snr_te == snr_val)[0]
    x = torch.from_numpy(X_te[idx]).to(DEVICE)
    cam, pred, p0 = grad_cam(model, x)
    T, k = x.shape[-1], int(frac * x.shape[-1])
    sal = np.zeros((len(idx), T), bool)
    np.put_along_axis(sal, np.argsort(-cam, axis=1)[:, :k], True, axis=1)
    rnd = np.zeros((len(idx), T), bool)
    rs = np.random.RandomState(seed)
    for i in range(len(idx)):
        rnd[i, rs.choice(T, k, replace=False)] = True

    def prob_after(mask):
        x2 = x.masked_fill(torch.from_numpy(mask).to(DEVICE)[:, None, :], 0.0)
        with torch.no_grad():
            model.eval()
            return model(x2).softmax(1).gather(1, pred[:, None]).mean().item()

    return dict(snr=snr_val, original=p0.gather(1, pred[:, None]).mean().item(),
                delete_salient=prob_after(sal), delete_random=prob_after(rnd))


del_rows = []
for name, runs in RUNS.items():
    for sv in (HIGH_SNR_DEMO, LOW_SNR_DEMO):
        r = deletion_test(runs[0]["model"], sv)
        del_rows.append(dict(model=name, **r))
deletion_df = pd.DataFrame(del_rows).round(3)
deletion_df.to_csv(OUT / "gradcam_deletion_test.csv", index=False)
print(deletion_df.to_string(index=False))

In [ ]:
att_model = RUNS["CNN+Attn"][0]["model"]
att_model.eval()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, sv in zip(axes, (HIGH_SNR_DEMO, LOW_SNR_DEMO)):
    idx = np.where(snr_te == sv)[0][:512]
    with torch.no_grad():
        att_model(torch.from_numpy(X_te[idx]).to(DEVICE))
    A = att_model.attn.last_attn.mean(0).cpu().numpy()
    im = ax.imshow(A, cmap="viridis"); ax.set_title(f"Mean attention matrix, SNR = {sv} dB")
    ax.set_xlabel("Key position"); ax.set_ylabel("Query position"); plt.colorbar(im, ax=ax)
savefig("attention_matrices.png")

In [ ]:
shutil.make_archive(str(BASE / "amc_outputs"), "zip", OUT)
print("Saved:", sorted(p.name for p in OUT.iterdir()))